In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Lặp đơn cho Hệ phi tuyến

Điều kiện hội tụ: 

* $G(D) \subset D, \forall X \in D $

* $||J|| \leq q < 1$ --> đảm bảo nghiệm duy nhất

Đánh giá sai số (hậu nghiệm) theo 1 trong 2 công thức sau:

* $||x_n - x^*|| \leq \dfrac{q^n}{1-q} ||x_1-x_0||$ $\quad$ (1)

* $||x_n - x^*|| \leq \dfrac{q}{1-q} ||x_n-x_{n-1}||$ $\quad$ (2)



# Thuật toán

1. **Khởi tạo**: 

- Chọn giá trị ban đầu $x_0$ và $y_0$.

- Tạo phương trình lặp $x = g_1(x)$ tương đương với phương trình $f_1(x, y) = 0$.

  Tạo phương trình lặp $y = g_2(y)$ tương đương với phương trình $f_2(x, y) = 0$.

2. **Lặp**: Thực hiện các bước sau

- Tính giá trị mới của $x$ và $y$:

  $x_{k+1} = g_1(x_k, y_k)$, $\quad\quad$ $y_{k+1} = g_2(x_k, y_k)$
      
- Kiểm tra điều kiện hội tụ: Dừng nếu đạt được độ chính xác mong muốn, hoặc đạt đến số lần lặp tối đa.

3. **Kết quả**: Nếu $\exists \lim\limits_{k \to \infty} x_k = \alpha$ và $g_1(x)$ liên tục thì $\alpha = g_1(\alpha)$. Tương tự $\beta = g_2(\beta)$

# Áp dụng

* Các chuẩn vecto: $||x||_{(1)} = \sum\limits_{i=1}^{k} |x_{i}|$, $\quad$ $||x||_{(\infty)} = \max\limits_{i} |x_{i}|$
* Các chuẩn ma trận (tính hệ số co):
    * Chuẩn max: $q = m * \max\limits_{i,j} |G_{ij}|$, $\quad$ (chuẩn vecto 1 và chuẩn vecto $\infty$)
    * Chuẩn cột: $q = ||G||_{(1)} = \max\limits_{1\leq j \leq n} \sum\limits_{i=1}^{m} |G_{ij}|$, $\quad$ (chuẩn vecto 1)
    * Chuẩn hàng: $q = ||G||_{(\infty)} = \max\limits_{1\leq i \leq m} \sum\limits_{j=1}^{n} |G_{ij}|$, $\quad$ (chuẩn vecto $\infty$)

## 1. Sai số tương đối

In [ ]:
def fixed_point_recursion_absolute (initial_values, q, eps, *funcs, norm):
    n = len(initial_values)
    values = np.array(initial_values)
    results = [[0] + initial_values]; i=0;

    new_eps = eps * (1-q)/ q
    print(f"new_eps: {new_eps}")

    while True:
        new_values = np.array([funcs[j](*values) for j in range(n)])
        
        # Calculate the differences
        diffs = np.abs(new_values - values)
        if (norm == 1): total_diff = np.sum(diffs)
        elif (norm == -1): total_diff = np.max(diffs)
        
        # Append the iteration number, new values, differences, and total_diff to the results list
        results.append([i+1] + new_values.tolist() + diffs.tolist() + [total_diff])
        values = new_values
        
        
        # Check for convergence
        if total_diff < new_eps:
            break
        i += 1  # Increment the iteration counter
        
    columns = ['Iteration'] + [f'x{j+1}' for j in range(n)] + [f'diff_x{j+1}' for j in range(n)] + ['total_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))  # Print the DataFrame without the index
    
    return values  

In [10]:
def g1(x, y, z):
    return 1/3 * (np.cos(y*z) + 0.5)

def g2(x, y, z):
    return 1/9 * np.sqrt(x**2 + np.sin(z) + 1.06) - 0.1

def g3(x, y, z):
    return -1/20 * np.exp(-x*y) - 9.1389/20

# Store g functions
def store_g_funcs(*g_funcs):
    return g_funcs

g_funcs = store_g_funcs(g1, g2, g3)

# Initial guess
initial_values = [0, 0, 0]
eps = 0.5 * 1e-7
q = 0.56 #column-norm

# Perform fixed-point iteration
solution = fixed_point_recursion_absolute(initial_values, q, eps, *g_funcs, norm=1)
print("Approximate solution:", solution)

new_eps: 3.928571428571427e-08
 Iteration                x1                x2                 x3           diff_x1           diff_x2           diff_x3        total_diff
         0 0.000000000000000 0.000000000000000  0.000000000000000               NaN               NaN               NaN               NaN
         1 0.500000000000000 0.014395890455411 -0.506945000000000 0.500000000000000 0.014395890455411 0.506945000000000 1.021340890455411
         2 0.499991123421941 0.000890556858707 -0.506586394896849 0.000008876578059 0.013505333596704 0.000358605103151 0.013872815277914
         3 0.499999966078184 0.000909195229164 -0.506922741429698 0.000008842656243 0.000018638370457 0.000336346532849 0.000363827559549
         4 0.499999964596468 0.000891745362941 -0.506922275286504 0.000000001481716 0.000017449866224 0.000000466143193 0.000017917491133
         5 0.499999965942465 0.000891770205638 -0.506922711336827 0.000000001345997 0.000000024842697 0.000000436050323 0.000000462239016
   

In [5]:
def g1(x, y, z):
    return 1/3 * (np.cos(y*z)) + 1/6

def g2(x, y, z):
    return -1/9 * np.sqrt(x**2 + np.sin(z) + 1.06) - 0.1

def g3(x, y, z):
    return -1/20 * np.exp(-x*y) -(10*np.pi-3)/60

# Store g functions
def store_g_funcs(*g_funcs):
    return g_funcs

g_funcs = store_g_funcs(g1, g2, g3)

# Initial guess
initial_values = [0, 0, 0]
eps = 1e-6
q = 0.416 #column-norm

# Perform fixed-point iteration
solution = fixed_point_recursion_absolute(initial_values, q, eps, *g_funcs, norm=1)
print("Approximate solution:", solution)

new_eps: 1.403846153846154e-06
 Iteration                x1                 x2                 x3           diff_x1           diff_x2           diff_x3        total_diff
         0 0.000000000000000  0.000000000000000  0.000000000000000               NaN               NaN               NaN               NaN
         1 0.500000000000000 -0.214395890455411 -0.523598775598299 0.500000000000000 0.214395890455411 0.523598775598299 1.237994666053710
         2 0.497901916407003 -0.200000000000000 -0.529256504413742 0.002098083592997 0.014395890455411 0.005657728815443 0.022151702863851
         3 0.498134326654338 -0.199567869406636 -0.528834138956606 0.000232410247334 0.000432130593364 0.000422365457135 0.001086906297834
         4 0.498145333571970 -0.199604819326524 -0.528824817282544 0.000011006917632 0.000036949919888 0.000009321674063 0.000057278511582
         5 0.498144712711300 -0.199605997697567 -0.528825955119565 0.000000620860670 0.000001178371043 0.000001137837021 0.000002937068

In [6]:
def g1(x, y, z):
    return (13 - y**2 + 1.5 * z**2) / 15

def g2(x, y, z):
    return (11 + z - x**2) / 10

def g3(x, y, z):
    return (20 + y**2) / 30

# Store g functions
def store_g_funcs(*g_funcs):
    return g_funcs

g_funcs = store_g_funcs(g1, g2, g3)

# Initial guess
initial_values = [0, 0, 0]
eps = 1 * 1e-6
q = 0.425 #column-norm

# Perform fixed-point iteration
solution = fixed_point_recursion_absolute(initial_values, q, eps, *g_funcs, norm=1)
print("Approximate solution:", solution)

new_eps: 1.352941176470588e-06
 Iteration                x1                x2                x3           diff_x1           diff_x2           diff_x3        total_diff
         0 0.000000000000000 0.000000000000000 0.000000000000000               NaN               NaN               NaN               NaN
         1 0.866666666666667 1.100000000000000 0.666666666666667 0.866666666666667 1.100000000000000 0.666666666666667 2.633333333333333
         2 0.830444444444444 1.091555555555555 0.707000000000000 0.036222222222222 0.008444444444445 0.040333333333333 0.085000000000001
         3 0.837218664609054 1.101736202469136 0.706383117695473 0.006774220164609 0.010180646913581 0.000616882304527 0.017571749382717
         4 0.835642866907777 1.100544802532571 0.707127421994370 0.001575797701277 0.001191399936565 0.000744304298897 0.003511501936739
         5 0.835922994934877 1.100882842098052 0.707039962079382 0.000280128027100 0.000338039565481 0.000087459914989 0.000705627507570
         6

## 2. Sai số tương đối:

In [7]:
def fixed_point_recursion_relative (initial_values, q, eta, *funcs, norm):
    n = len(initial_values)
    values = np.array(initial_values)
    results = [[0] + initial_values]; i=0;

    new_eta = eta * (1-q)/ q
    print(f"new_eta: {new_eta}")

    while True:
        new_values = np.array([funcs[j](*values) for j in range(n)])
        
        # Calculate the differences
        diffs = np.abs(new_values - values)
        itself = np.abs(new_values)
        if (norm == 1): 
            total_diff = np.sum(diffs)
            total_itself = np.sum(itself)
        elif (norm == -1): 
            total_diff = np.max(diffs)
            total_itself = np.max(itself)
       
        if (i!=0):
            relative_diff = total_diff / total_itself
        else:
            relative_diff = None
        
        # Append the iteration number, new values, differences, and total_diff to the results list
        results.append([i+1] + new_values.tolist() + diffs.tolist() + [total_diff] + [relative_diff])
        values = new_values
        
        # Check for convergence
        if i!=0 and relative_diff < new_eta:
            break    

        i += 1  # Increment the iteration counter 
    
    columns = ['Iteration'] + [f'x{j+1}' for j in range(n)] + [f'diff_x{j+1}' for j in range(n)] + ['total_diff'] + ['relative_diff']
    df = pd.DataFrame(results, columns=columns)
    print(df.to_string(index=False))  # Print the DataFrame without the index
    
    return values

In [8]:
def g1(x, y):
    return 1/3 * np.sin(x*y)

def g2(x, y):
    return 1/4 * np.cos(x**2+y**2)

# Store g functions
def store_g_funcs(*g_funcs):
    return g_funcs

g_funcs = store_g_funcs(g1, g2)

# Initial guess
initial_values = [1, 1]
eta = 1 * 1e-4
q = 1/3 * np.cos(1) + 1/2 * np.sin(2) #column-norm

# Perform fixed-point iteration
solution = fixed_point_recursion_relative (initial_values, q, eta, *g_funcs, norm=1)
print("Approximate solution:", solution)

new_eta: 5.75424680604918e-05
 Iteration                 x1                 x2           diff_x1           diff_x2        total_diff     relative_diff
         0  1.000000000000000  1.000000000000000               NaN               NaN               NaN               NaN
         1  0.280490328269299 -0.104036709136786 0.719509671730701 1.104036709136786 1.823546380867487               NaN
         2 -0.009725716443514  0.248999421334286 0.290216044712812 0.353036130471071 0.643252175183884 2.486237637010470
         3 -0.000807231799814  0.249518176542058 0.008918484643700 0.000518755207772 0.009437239851472 0.037699887973752
         4 -0.000067139668458  0.249515618482908 0.000740092131355 0.000002558059150 0.000742650190505 0.002975566886133
         5 -0.000005584131966  0.249515648405069 0.000061555536492 0.000000029922161 0.000061585458653 0.000246814501623
         6 -0.000000464442769  0.249515648242439 0.000005119689197 0.000000000162630 0.000005119851827 0.000020519123083
Ap